# Sample-wise gradients for BatchNorm layers

The Goal of this notebook is to develop sample-wise gradients that work with BatchNorm layers.

We quantify the functionality by comparing the mean of sample-wise gradients to the batch gradient, which is the expected behavior.

In [1]:
import torch
import torch.nn as nn
from torch.func import functional_call, vmap, grad
from copy import deepcopy

from examples.models import SimpleMLP, BatchNormMLP, ClassificationModule
from perspic.analyzer import SamplewiseCalculatorFunctorch
from perspic.calculator.samplewise import SamplewiseCalculatorFunctorch

samplewise_grap_fn = SamplewiseCalculatorFunctorch()._compute_per_sample_gradient_network_sum

# Print whether cuda is used
use_cuda = torch.cuda.is_available()
# If cuda is available, use CPU
device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Set up simple Model and Data in CIFAR10 input dimensions

seed = 42
torch.manual_seed(seed)

model = BatchNormMLP()
# model = SimpleMLP()

data = torch.randn(8, 3, 32, 32)  # Batch of 8 CIFAR10-like images
labels = torch.randint(0, 1, (8,))  # Random labels for 1 class

loss = nn.CrossEntropyLoss()

# Check forward pass
outputs = model(data)
print("Output shape:", outputs.shape)  # Should be (8, 10)

# Check loss computation
computed_loss = loss(outputs, labels)
print("Computed loss:", computed_loss.item())

Output shape: torch.Size([8, 1])
Computed loss: 0.0


In [3]:
# Import torch nn BatchNorm2d for reference
from torch.nn import BatchNorm2d

isinstance(nn.BatchNorm1d(3), nn.modules.batchnorm._BatchNorm)  # Should return True

True

In [4]:
class BatchStatSnapshot:
    """
    Context manager that forces a model's running stats to match 
    the CURRENT batch stats exactly, then temporarily switches to eval mode.
    
    This allows vmap to compute per-sample gradients using the 
    current batch's normalization (simulating train mode) 
    without the mathematical coupling errors.
    """
    def __init__(self, model, data):
        self.model = model
        self.data = data
        self.original_momentums = {}
        self.original_training_states = {}

    def __enter__(self):
        # 1. Save original states and force momentum to 1.0
        # Momentum = 1.0 means the running stats will be instantly overwritten 
        # by the current batch's stats (ignoring history).
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                self.original_momentums[name] = module.momentum
                self.original_training_states[name] = module.training
                module.momentum = 1.0
                # module.momentum = 0.5
                module.train() # Ensure we are in train mode to update stats

        # 2. Perform a dummy forward pass to update the buffers
        # We use no_grad because we only care about updating the running_mean/var buffers
        with torch.no_grad():
            self.model(self.data)

        # 3. Switch to Eval mode
        # Now 'running_mean' == 'current_batch_mean'.
        # In Eval mode, PyTorch uses these buffers as fixed constants.
        # This satisfies vmap's independence requirement.
        self.model.eval()
        
        return self.model

    def __exit__(self, exc_type, exc_value, traceback):
        # 4. Restore original states
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                module.momentum = self.original_momentums[name]
                module.train(self.original_training_states[name])

In [5]:
# Test whether snapshot works correctly

# Update the model on some data to have non-trivial running stats
model.train()
data_intermediate = torch.randn(8, 3, 32, 32)  # Larger batch to update stats
out = model(data_intermediate)  # Forward pass to update running stats


snapshot_dict = {}
with BatchStatSnapshot(model, data):
    # Inside this block, model should have running stats 
    # equal to the current batch stats and be in eval mode.
    for name, module in model.named_modules():
        if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            snapshot_dict[name] = {
                "training": module.training,  # Should be False
                "momentum": module.momentum,  # Should be 1.0
                "running_mean": module.running_mean.clone(),
                "running_var": module.running_var.clone()
            }

# Compare to regular train model
train_dict = {}
model.train()
out = model(data)  # Forward pass to update running stats
for name, module in model.named_modules():
    if isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
        train_dict[name] = {
            "training": module.training,  # Should be True
            "momentum": module.momentum,  # Original momentum
            "running_mean": module.running_mean.clone(),
            "running_var": module.running_var.clone()
        }

# Now compare snapshot_dict and train_dict
for name in snapshot_dict.keys():
    snap = snapshot_dict[name]
    train = train_dict[name]
    print(f"Module: {name}")
    print(f"  Training mode - Snapshot: {snap['training']}, Train: {train['training']}")
    print(f"  Momentum - Snapshot: {snap['momentum']}, Train: {train['momentum']}")
    print(f"  Running Mean equal: {torch.allclose(snap['running_mean'], train['running_mean'], atol=1e-8)}")
    print(f"  Running Var equal: {torch.allclose(snap['running_var'], train['running_var'], atol=1e-8)}")
    print()

Module: model.3
  Training mode - Snapshot: False, Train: True
  Momentum - Snapshot: 1.0, Train: 0.1
  Running Mean equal: True
  Running Var equal: True



In [6]:
# Now compute the sample-wise gradients using the BatchStatSnapshot context manager



def compute_samplewise_grads(model, data):
    with BatchStatSnapshot(model, data) as model_in_snapshot:

        # Compute gradients per sample
        data = data.unsqueeze(1)  # Add channel dimension if needed
        samplewise_grads = samplewise_grap_fn(model_in_snapshot, data)
    
    return samplewise_grads

samplewise_grads = compute_samplewise_grads(model, data)

for name, grad in samplewise_grads.items():
    print(f"Parameter: {name}, Sample-wise grad shape: {grad.shape}")

# Compute regular model jacobian for comparison
model.zero_grad()
outputs = model(data)
regular_jacobian = {}
for i in range(outputs.shape[1]):  # For each class output
    model.zero_grad()
    outputs[:, i].sum().backward(retain_graph=True)
    for name, param in model.named_parameters():
        if name not in regular_jacobian:
            regular_jacobian[name] = []
        regular_jacobian[name].append(param.grad.detach().clone())
for name in regular_jacobian:
    regular_jacobian[name] = torch.stack(regular_jacobian[name], dim=0)  # Shape: (num_classes, ...)    

print("\nRegular Jacobian shapes:")
for name, jac in regular_jacobian.items():
    print(f"Parameter: {name}, Jacobian shape: {jac.shape}")



total_norm_samplewise = 0.0
total_norm_regular = 0.0
per_sample_gradient_norm = 0.0 # Computing the norm BEFORE averaging

# Compare contents of sample-wise grads and regular jacobian
print("\nComparing Sample-wise Grads to Regular Jacobian:")
for name in regular_jacobian:
    sample_grad = samplewise_grads[name]  # Shape: (batch_size, param_shape...)
    reg_jac = regular_jacobian[name]      # Shape: (num_classes, param_shape...)
    
    per_sample_gradient_norm += (sample_grad ** 2).sum()

    # For simplicity, we can compare the norms
    sample_grad_sum = sample_grad.sum(dim=(0, 1))  # Sum over batch dimension

    # subtract both tensors
    assert sample_grad_sum.shape == reg_jac.shape, "Shapes do not match for comparison"
    diff = sample_grad_sum - reg_jac
    print(f"Parameter: {name}, Difference norm: {diff.norm()}")
    
    # Compute the error in the norms
    sample_wise_norm = (sample_grad_sum ** 2).sum()
    reg_jac_norm = (reg_jac ** 2).sum()
    error = torch.abs(sample_wise_norm - reg_jac_norm) / (reg_jac_norm + 1e-8)
    print(f"Sample-wise norm: {sample_wise_norm}, Regular Jacobian norm: {reg_jac_norm}, Relative error: {error}\n")
    total_norm_samplewise += sample_wise_norm
    total_norm_regular += reg_jac_norm

print(f"Total Sample-wise norm: {total_norm_samplewise}, Total Regular Jacobian norm: {total_norm_regular}, Relative in averaged gradient norm: {torch.abs(total_norm_samplewise - total_norm_regular) / (total_norm_regular + 1e-8)}")
print(rf"Per-sample Gradient norm (before averaging): {per_sample_gradient_norm}")


Parameter: model.0.weight, Sample-wise grad shape: torch.Size([8, 1, 1, 128, 3072])
Parameter: model.0.bias, Sample-wise grad shape: torch.Size([8, 1, 1, 128])
Parameter: model.2.weight, Sample-wise grad shape: torch.Size([8, 1, 1, 128, 128])
Parameter: model.2.bias, Sample-wise grad shape: torch.Size([8, 1, 1, 128])
Parameter: model.3.weight, Sample-wise grad shape: torch.Size([8, 1, 1, 128])
Parameter: model.3.bias, Sample-wise grad shape: torch.Size([8, 1, 1, 128])
Parameter: model.5.weight, Sample-wise grad shape: torch.Size([8, 1, 1, 1, 128])
Parameter: model.5.bias, Sample-wise grad shape: torch.Size([8, 1, 1, 1])

Regular Jacobian shapes:
Parameter: model.0.weight, Jacobian shape: torch.Size([1, 128, 3072])
Parameter: model.0.bias, Jacobian shape: torch.Size([1, 128])
Parameter: model.2.weight, Jacobian shape: torch.Size([1, 128, 128])
Parameter: model.2.bias, Jacobian shape: torch.Size([1, 128])
Parameter: model.3.weight, Jacobian shape: torch.Size([1, 128])
Parameter: model.3.

Gradients differ in all layers before the first BatchNorm layer. 

This is due to the fact that BatchNorm layers compute statistics (mean and variance) over the batch, which affects the forward pass and consequently the gradients. While fixing the running statistics gives the same output in the forward pass, however, the the constant $\mu$ and $\sigma$ kill gradients flowing back through them. 
This means 
\begin{align}
\frac{\partial f(x)}{\partial \theta} = \frac{\partial \tilde{f}(x)}{\partial \theta} + \frac{\partial {f(x)}}{\partial \mu} \frac{\partial \mu}{\partial \theta} + \frac{\partial {f(x)}}{\partial \sigma} \frac{\partial \sigma}{\partial \theta}
\end{align}
where $f(x)$ is the network function with BatchNorm layers computing batch statistics, and $\tilde{f}(x)$ is the network function with BatchNorm layers using fixed statistics. The second and third terms are missing in the sample-wise gradient computation, leading to discrepancies in gradients before the first BatchNorm layer.

In [7]:

break 

# --- USAGE EXAMPLE ---

# 1. Setup Standard Model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=False).to(device)
data = torch.randn(4, 3, 224, 224, device=device)
targets = torch.randint(0, 1000, (4,), device=device)
criterion = nn.CrossEntropyLoss()

# 2. Define per-sample loss function
def compute_loss(params, buffers, sample, target):
    batch_sample = sample.unsqueeze(0)
    batch_target = target.unsqueeze(0)
    predictions = functional_call(model, (params, buffers), (batch_sample,))
    return criterion(predictions, batch_target)

# 3. EXECUTION
params = dict(model.named_parameters())
buffers = dict(model.named_buffers())

# Wrap the computation in the Snapshot context
with BatchStatSnapshot(model, data):
    # We must refresh buffers because the context manager just updated them!
    updated_buffers = dict(model.named_buffers())
    
    # Define gradient function
    compute_grad = grad(compute_loss, argnums=0)
    compute_sample_grads = vmap(compute_grad, in_dims=(None, None, 0, 0))
    
    # Compute!
    per_sample_grads = compute_sample_grads(params, updated_buffers, data, targets)

print("Success! Per-sample gradients computed using current batch stats.")
print(f"Gradient shape for first layer: {per_sample_grads['conv1.weight'].shape}")

SyntaxError: 'break' outside loop (3047869910.py, line 1)